# Differential Invariants and Reduction of Order

This tutorial demonstrates two focused calculations based on **F. Güngör** (*Lie symmetry group methods for differential equations*, arXiv:1901.01543):

1. **Reduction of Order for a Fiber-Preserving ODE**: $\ddot{y} = \frac{\dot{y}^2}{y} - \dot{y}$, including symmetry verification and an exact solution.
2. **Euclidean Motion Group $E(2)$ and Curvature**: verification that a circle has the constant differential invariant $\kappa^2=1/R^2$.

In [ ]:
import sympy as sp

from symlie import (
    infinitesimals,
    lie_bracket,
    max_derivative_order,
)

sp.init_printing()

x = sp.symbols("x")
y = sp.Function("y")(x)

# Fiber-preserving ODE: y'' = y'^2 / y - y'
fiber_ode = y.diff(x, 2) - y.diff(x) ** 2 / y + y.diff(x)
print("ODE Order:", max_derivative_order(fiber_ode, y, x))
sp.Eq(fiber_ode, 0)

## 1. Lie Point Symmetries and Commutators of the Fiber ODE

The equation admits the 2-dimensional abelian Lie algebra $A_{2,1}$ generated by translations $\mathbf{v}_1 = \partial_x$ and scaling $\mathbf{v}_2 = y\partial_y$.

In [ ]:
sol_fiber = infinitesimals(fiber_ode, y, x, ansatz_degree=1)
print(f"Dimension within degree-1 ansatz: {sol_fiber.ansatz_dimension}\n")

v1, v2 = sol_fiber.basis
print("v_1 (Translation in x):", v1)
print("v_2 (Scaling in y):    ", v2)

bracket_12 = lie_bracket(v1, v2, y, x)
print("[v_1, v_2] =", bracket_12)
assert bracket_12.xi == (0,) and bracket_12.phi == (0,)
print("Commutator verified: [v_1, v_2] = 0 (Abelian algebra A_2,1)")

## 2. Step-by-Step Reduction of Order to Quadrature

Using the symmetry $\mathbf{v}_1 = \partial_x$, we introduce the differential invariant $u(y) = y'(x)$.
Then $y''(x) = u \frac{du}{dy}$, reducing the 2nd-order ODE to a 1st-order separable equation:
$$u \frac{du}{dy} - \frac{u^2}{y} + u = 0 \implies \frac{du}{dy} - \frac{u}{y} = -1$$

Solving gives $u(y) = y'(x) = c_1 y - y \ln y$, which is integrated again to obtain the general analytical solution:
$$y(x) = e^{1 - c_1 + c_2 e^{-x}}$$

In [ ]:
c1, c2 = sp.symbols("c1 c2")
exact_sol = sp.exp(1 - c1 + c2 * sp.exp(-x))

residual = sp.simplify(fiber_ode.subs(y, exact_sol).doit())
print("Residual of exact solution:", residual)
assert residual == 0
print("Verification: Exact solution satisfies the ODE identically!")

## 3. Euclidean Curvature Invariant $E(2)$

Under the planar Euclidean motion group $E(2)$:
$$\mathbf{v}_1 = \partial_x, \quad \mathbf{v}_2 = \partial_y, \quad \mathbf{v}_3 = -y\partial_x + x\partial_y$$

The second-order differential invariant is the curvature $\kappa(x) = \frac{y''(x)}{(1 + y'(x)^2)^{3/2}}$.

Circles of constant curvature $\kappa^2 = 1/R^2$ satisfy $(x - x_0)^2 + (y - y_0)^2 = R^2$.

In [ ]:
R, x0, y0 = sp.symbols("R x0 y0", positive=True)
circle_y = y0 + sp.sqrt(R**2 - (x - x0) ** 2)

# Compute curvature squared kappa^2 = y''^2 / (1 + y'^2)^3
y_p = circle_y.diff(x)
y_pp = circle_y.diff(x, 2)
kappa_sq = sp.simplify(y_pp**2 / (1 + y_p**2) ** 3)

print("Computed Curvature Squared kappa^2:", kappa_sq)
assert kappa_sq == 1 / R**2
print("Confirmed: Circle has constant invariant curvature |kappa| = 1/R!")